In [1]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, Dropout, Input
from keras.models import load_model
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import pickle

# Set seed for reproducibility
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# Dummy amino acid sequences for demonstration
sequences = [
    "ACDEFGHIKLMNPQRSTVWY",
    "AGGAGGAGGAGG",
    "MKTFFVVLAVAA",
    "LLKLLKKLLKKL",
    "GASGDLGASGDL",
    "RWRRWRRWRRW",
    "QQQQQQQQQQQQ",
    "NYNYNYNYNYNY",
    "PPPPPPPPPPPP",
    "VVVVVVVVVVVV"
]

# Biological feature extraction function
def extract_features(seq):
    analyzed_seq = ProteinAnalysis(seq)
    return [
        analyzed_seq.molecular_weight(),
        analyzed_seq.aromaticity(),
        analyzed_seq.instability_index(),
        analyzed_seq.isoelectric_point(),
        analyzed_seq.gravy()
    ]

# Feature labels for interpretation later
feature_names = [
    "Molecular Weight",
    "Aromaticity",
    "Instability Index",
    "Isoelectric Point",
    "Hydrophobicity (GRAVY)"
]

# Extract features and generate meaningful dummy target
X = []
y = []

for seq in sequences:
    features = extract_features(seq)
    X.append(features)

    # Create a synthetic "biological score" from features
    score = (
        0.1 * features[0] +  # weight
        5.0 * features[1] +  # aromaticity
        -0.2 * features[2] +  # instability
        0.3 * features[3] +  # isoelectric point
        1.5 * features[4]    # gravy
    ) + random.gauss(0, 1)  # add some noise
    y.append(score)

# Convert to numpy
X = np.array(X)
y = np.array(y)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save the scaler to use during inference
with open("scaler_model3.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Build a better regression model
def build_model_3(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='linear')
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Create and train model
print("🧠 Training Model 3...")
model_3 = build_model_3(X.shape[1])
early_stop = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

model_3.fit(X_train, y_train, epochs=100, batch_size=8, callbacks=[early_stop], verbose=1)

# Save model
model_3.save("model 3.h5")
print("✅ Model saved as Model_3.h5")

🧠 Training Model 3...
Epoch 1/100
1/1 [==============================] - 0s 428ms/step - loss: 21469.2422 - mae: 141.7192
Epoch 2/100
1/1 [==============================] - 0s 5ms/step - loss: 21493.1992 - mae: 141.7475
Epoch 3/100
1/1 [==============================] - 0s 6ms/step - loss: 21489.7891 - mae: 141.7872
Epoch 4/100
1/1 [==============================] - 0s 7ms/step - loss: 21390.0391 - mae: 141.3788
Epoch 5/100
1/1 [==============================] - 0s 5ms/step - loss: 21429.1543 - mae: 141.5452
Epoch 6/100
1/1 [==============================] - 0s 4ms/step - loss: 21401.8027 - mae: 141.4216
Epoch 7/100
1/1 [==============================] - 0s 6ms/step - loss: 21388.3516 - mae: 141.3661
Epoch 8/100
1/1 [==============================] - 0s 4ms/step - loss: 21404.7852 - mae: 141.4535
Epoch 9/100
1/1 [==============================] - 0s 4ms/step - loss: 21395.9590 - mae: 141.4240
Epoch 10/100
1/1 [==============================] - 0s 8ms/step - loss: 21356.2070 - mae: 141.